# Exploración del Dataset Olist - Brazilian E-Commerce

Análisis exploratorio inicial de los 9 archivos CSV del dataset público de Olist en Kaggle.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_PATH = '../data/raw/'

## 1) Carga de datos

In [ ]:
orders        = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'])
customers     = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
order_items   = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv', parse_dates=['shipping_limit_date'])
payments      = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
reviews       = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products      = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
sellers       = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
geolocation   = pd.read_csv(DATA_PATH + 'olist_geolocation_dataset.csv')
category_xlat = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

tables = {
    'orders':        orders,
    'customers':     customers,
    'order_items':   order_items,
    'payments':      payments,
    'reviews':       reviews,
    'products':      products,
    'sellers':       sellers,
    'geolocation':   geolocation,
    'category_xlat': category_xlat,
}

print('Tablas cargadas:', list(tables.keys()))

## 2) Estructura

In [ ]:
for name, df in tables.items():
    print(f'\n{'='*55}')
    print(f'  {name}  —  shape: {df.shape}')
    print(f'{'='*55}')
    print(df.dtypes.to_string())

## 3) Valores nulos

In [ ]:
for name, df in tables.items():
    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    null_pct = null_pct[null_pct > 0]  # mostrar solo columnas con nulos
    print(f'\n{'='*55}')
    print(f'  {name}  —  % nulos por columna')
    print(f'{'='*55}')
    if null_pct.empty:
        print('  Sin valores nulos.')
    else:
        print(null_pct.to_string())

## 4) Vista previa

In [ ]:
for name, df in tables.items():
    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    display(df.head(3))

## 5) Distribución temporal

In [ ]:
orders_valid = orders[orders['order_purchase_timestamp'].notna()].copy()
orders_valid['year_month'] = orders_valid['order_purchase_timestamp'].dt.to_period('M')

monthly_counts = (
    orders_valid
    .groupby('year_month')
    .size()
    .reset_index(name='num_orders')
    .sort_values('year_month')
)
monthly_counts['year_month_str'] = monthly_counts['year_month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(
    data=monthly_counts,
    x='year_month_str',
    y='num_orders',
    color='steelblue',
    ax=ax
)
ax.set_title('Cantidad de pedidos por mes', fontsize=14, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Número de pedidos')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 6) Distribución geográfica

In [ ]:
orders_customers = orders[['order_id', 'customer_id']].merge(
    customers[['customer_id', 'customer_state']],
    on='customer_id',
    how='left'
)

top10_states = (
    orders_customers
    .groupby('customer_state')
    .size()
    .reset_index(name='num_orders')
    .sort_values('num_orders', ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=top10_states,
    x='customer_state',
    y='num_orders',
    palette='Blues_r',
    ax=ax
)
ax.set_title('Top 10 estados por cantidad de pedidos', fontsize=14, fontweight='bold')
ax.set_xlabel('Estado')
ax.set_ylabel('Número de pedidos')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()